### 🌲 Introducción al Random Forest

El **Random Forest** (bosque aleatorio) es un algoritmo de *ensemble learning* que combina las predicciones de muchos árboles de decisión para obtener un modelo más robusto y preciso que un único árbol.

**¿Cómo funciona?**

- **Bagging (Bootstrap Aggregating):** cada árbol del bosque se entrena con una muestra aleatoria (con reemplazo) del conjunto de entrenamiento original, de modo que ningún árbol ve exactamente los mismos datos.
- **Selección aleatoria de variables:** en cada división (split) de cada árbol, solo se considera un subconjunto aleatorio de las variables disponibles, en lugar de todas. Esto decorrelaciona los árboles entre sí.
- **Votación / promedio:** para clasificación, la predicción final se obtiene por **voto mayoritario** entre todos los árboles; para regresión, se promedian sus salidas.

**Diferencias clave con el Decision Tree:**

| Aspecto | Decision Tree | Random Forest |
|---|---|---|
| Número de modelos | Un único árbol | Muchos árboles combinados (ensemble) |
| Varianza | Alta: sensible a pequeños cambios en los datos, tiende al sobreajuste | Baja: al promediar/votar entre árboles decorrelacionados, se reduce el sobreajuste |
| Datos de entrenamiento por árbol | Usa todo el conjunto de entrenamiento | Cada árbol usa una muestra bootstrap distinta |
| Variables consideradas en cada split | Todas las variables disponibles | Un subconjunto aleatorio de variables |
| Interpretabilidad | Alta: se puede visualizar y seguir el árbol completo (`plot_tree`) | Menor: no se puede visualizar un único árbol representativo, aunque sigue permitiendo calcular `feature_importances_` (promediadas entre todos los árboles) |
| Estabilidad | Inestable ante ruido/outliers | Más estable y generalmente con mejor rendimiento predictivo |
| Costo computacional | Bajo | Mayor (se entrenan y evalúan múltiples árboles), aunque es paralelizable |

**En resumen:** mientras que el Decision Tree busca la estructura óptima de un único árbol (lo que lo hace fácil de interpretar pero propenso a sobreajustar), el Random Forest sacrifica algo de interpretabilidad para ganar robustez, promediando el "voto" de muchos árboles distintos entrenados sobre variaciones de los datos y las variables.

A continuación entrenaremos y evaluaremos un Random Forest Classifier sobre el mismo problema de clasificación de abandono de clientes (churn) que trabajamos con el Decision Tree, para comparar su desempeño.

## 1. Carga de los datos

Cargamos la misma base de datos de abandono de clientes (*customer churn*) utilizada en el notebook de árboles de decisión. Así, este notebook se puede ejecutar de forma independiente.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/IBM/watson-studio-learning-path-assets/master/data/customer-churn-kaggle.csv"
df = pd.read_csv(url)

print(f"DataFrame cargado: {df.shape[0]} filas y {df.shape[1]} columnas.")

DataFrame cargado: 3333 filas y 21 columnas.


## 2. Información de las columnas

Esta salida muestra el nombre de cada columna, su tipo de dato, cuántos valores no nulos contiene y el uso de memoria.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   state                   3333 non-null   str    
 1   account length          3333 non-null   int64  
 2   area code               3333 non-null   int64  
 3   phone number            3333 non-null   str    
 4   international plan      3333 non-null   str    
 5   voice mail plan         3333 non-null   str    
 6   number vmail messages   3333 non-null   int64  
 7   total day minutes       3333 non-null   float64
 8   total day calls         3333 non-null   int64  
 9   total day charge        3333 non-null   float64
 10  total eve minutes       3333 non-null   float64
 11  total eve calls         3333 non-null   int64  
 12  total eve charge        3333 non-null   float64
 13  total night minutes     3333 non-null   float64
 14  total night calls       3333 non-null   int64  
 15

## 3. Resumen estadístico con `describe`

Usamos `include="all"` para que el resumen abarque también las columnas categóricas y la variable booleana, además de las numéricas.

In [3]:
df.describe(include="all")

,state,account length,area code,phone number,international plan,voice mail plan,number vmail messages,total day minutes,total day calls,total day charge,...,total eve calls,total eve charge,total night minutes,total night calls,total night charge,total intl minutes,total intl calls,total intl charge,customer service calls,churn
count,3333,3333.000000,3333.000000,3333,3333,3333,3333.000000,3333.000000,3333.000000,3333.000000,...,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333.000000,3333
unique,51,NaN,NaN,3333,2,2,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,WV,NaN,NaN,382-4657,no,no,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
freq,106,NaN,NaN,1,3010,2411,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2850
mean,NaN,101.064806,437.182418,NaN,NaN,NaN,8.099010,179.775098,100.435644,30.562307,...,100.114311,17.083540,200.872037,100.107711,9.039325,10.237294,4.479448,2.764581,1.562856,NaN
std,NaN,39.822106,42.371290,NaN,NaN,NaN,13.688365,54.467389,20.069084,9.259435,...,19.922625,4.310668,50.573847,19.568609,2.275873,2.791840,2.461214,0.753773,1.315491,NaN
min,NaN,1.000000,408.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,23.200000,33.000000,1.040000,0.000000,0.000000,0.000000,0.000000,NaN
25%,NaN,74.000000,408.000000,NaN,NaN,NaN,0.000000,143.700000,87.000000,24.430000,...,87.000000,14.160000,167.000000,87.000000,7.520000,8.500000,3.000000,2.300000,1.000000,NaN
50%,NaN,101.000000,415.000000,NaN,NaN,NaN,0.000000,179.400000,101.000000,30.500000,...,100.000000,17.120000,201.200000,100.000000,9.050000,10.300000,4.000000,2.780000,1.000000,NaN
75%,NaN,127.000000,510.000000,NaN,NaN,NaN,20.000000,216.400000,114.000000,36.790000,...,114.000000,20.000000,235.300000,113.000000,10.590000,12.100000,6.000000,3.270000,2.000000,NaN


## 4. Valores nulos y cantidad de atributos

Contamos los nulos de cada columna y el total. También mostramos las dimensiones del DataFrame: filas (registros) y columnas (atributos).

In [4]:
nulos_por_columna = df.isna().sum()
print("Valores nulos por columna:")
display(nulos_por_columna)
print(f"\nTotal de valores nulos: {nulos_por_columna.sum()}")
print(f"Cantidad de filas: {df.shape[0]}")
print(f"Cantidad de columnas o atributos: {df.shape[1]}")

Valores nulos por columna:


state                     0
account length            0
area code                 0
phone number              0
international plan        0
voice mail plan           0
number vmail messages     0
total day minutes         0
total day calls           0
total day charge          0
total eve minutes         0
total eve calls           0
total eve charge          0
total night minutes       0
total night calls         0
total night charge        0
total intl minutes        0
total intl calls          0
total intl charge         0
customer service calls    0
churn                     0
dtype: int64


Total de valores nulos: 0
Cantidad de filas: 3333
Cantidad de columnas o atributos: 21


## 5. Registros duplicados

Comprobamos cuántas filas repetidas hay en el DataFrame.

In [5]:
cantidad_duplicados = df.duplicated().sum()
print(f"Cantidad de registros duplicados: {cantidad_duplicados}")
print("¿Hay registros duplicados?", "Sí" if cantidad_duplicados > 0 else "No")

Cantidad de registros duplicados: 0
¿Hay registros duplicados? No


## 6. Atributo objetivo de la clasificación

El atributo objetivo recomendado es **`churn`**. Indica si el cliente abandonó (`True`) o permaneció (`False`), por lo que representa la clase que queremos predecir. Las demás columnas serán candidatas a variables predictoras; `phone number` es un identificador y no conviene usarlo como predictor.

In [6]:
atributo_objetivo = "churn"
print(f"Atributo objetivo seleccionado: {atributo_objetivo}")
print(f"Tipo de dato del objetivo: {df[atributo_objetivo].dtype}")

Atributo objetivo seleccionado: churn
Tipo de dato del objetivo: bool


## 7. Cantidad de casos por clase

Contamos por separado cuántos clientes permanecen (`False`) y cuántos abandonan (`True`).

In [7]:
conteo_clases = df[atributo_objetivo].value_counts().reindex([False, True], fill_value=0)
conteo_clases.index = ["False (permanece)", "True (abandona)"]
display(conteo_clases)

False (permanece)    2850
True (abandona)       483
Name: count, dtype: int64